In [0]:
%run ../utils/utils_feat_squad2_99_helpers

In [0]:
inicio = log_inicio("feat_squad2_01_connection_sql")

# ─────────────────────────────────────────────
# 1. TESTAR CONEXÃO
# ─────────────────────────────────────────────

try:
    df_test = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", "SELECT 1 AS conexao_ok")
        .load()
    )

    display(df_test)
    log.info("Conexão com SQL Server validada!")

except Exception as e:
    log.error(f"Erro na conexão: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 2. VERIFICAR SCHEMAS DISPONÍVEIS
# ─────────────────────────────────────────────

try:
    df_schemas = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT name AS schema_name
            FROM sys.schemas
            WHERE name NOT IN (
                'sys', 'INFORMATION_SCHEMA',
                'db_owner', 'db_accessadmin',
                'db_securityadmin', 'db_ddladmin',
                'db_backupoperator', 'db_datareader',
                'db_datawriter', 'db_denydatareader',
                'db_denydatawriter'
            )
        """)
        .load()
    )

    log.info("Schemas disponíveis:")
    display(df_schemas)

except Exception as e:
    log.error(f"Erro ao listar schemas: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 3. VERIFICAR TABELAS DO SQUAD 2
# ─────────────────────────────────────────────

try:
    df_tabelas = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT
                s.name AS schema_name,
                t.name AS table_name
            FROM sys.tables t
            JOIN sys.schemas s
                ON t.schema_id = s.schema_id
            WHERE s.name IN ('dbo', 'squad2')
            AND (
                t.name LIKE '%squad2%'
                OR t.name LIKE '%rastreamento%'
                OR t.name LIKE '%ecommerce%'
            )
        """)
        .load()
    )

    log.info("Tabelas relacionadas ao Squad 2 no banco:")
    display(df_tabelas)

except Exception as e:
    log.error(f"Erro ao listar tabelas: {str(e)}")
    raise


# ─────────────────────────────────────────────
# 4. VERIFICAR STATUS DO SCHEMA SQUAD2
# ─────────────────────────────────────────────

try:
    df_schema = (
        spark.read
        .format("sqlserver")
        .options(**SQL_OPTIONS)
        .option("query", """
            SELECT
                CASE
                    WHEN EXISTS (
                        SELECT 1
                        FROM sys.schemas
                        WHERE name = 'squad2'
                    )
                    THEN 'DISPONIVEL'
                    ELSE 'PENDENTE_CRIACAO'
                END AS squad2_status
        """)
        .load()
    )

    status = df_schema.collect()[0]["squad2_status"]

    if status == "DISPONIVEL":
        log.info("Schema squad2 disponível — pronto para uso.")
    else:
        log.warning(
            "Schema squad2 pendente — "
            "verifique com o responsável antes de gravar as tabelas."
        )

    display(df_schema)

except Exception as e:
    log.error(f"Erro ao verificar schema: {str(e)}")
    raise


log_fim("feat_squad2_01_connection_sql", inicio)